# RobustCE: practical experiments

## 1. Checking the analytic loss against numerical maximization

We compare the specified binary RobustCE with a direct numerical maximization of CE over the feasible target-probability interval. Model training will be added in the next step.

Notation: $y\in\{0,1\}$ is the observed label, $z\in\mathbb R$ is the logit, and $\sigma(z)$ is the predicted probability of class 1. We set the radius $\delta\ge0$ directly.

All computations use **PyTorch on CPU**. We use `float64` for this numerical check. The required packages are `torch`, `plotly`, and `IPython`.


In [1]:
from itertools import product

import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

DEVICE = torch.device("cpu")
DTYPE = torch.float64

# Use one thread for these small grids; this check involves no randomness.
torch.set_num_threads(1)
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"

print(f"PyTorch {torch.__version__} | device={DEVICE} | dtype={DTYPE}")


PyTorch 2.11.0+cpu | device=cpu | dtype=torch.float64


### Analytic formula

We use the specified loss:

$$
L_{\mathrm{RobustCE}}(z,y;\delta)
=\underbrace{\operatorname{softplus}(z)-yz}_{L_{\mathrm{CE}}(z,y)}
+\underbrace{\hat\delta\,\operatorname{ReLU}((2y-1)z)}_{R(z,y;\delta)},
\qquad \hat\delta=\min(1,\delta).
$$

The functions below return a loss **per example**, without averaging. During subsequent training, we can use `binary_robust_ce(logits, targets, delta).mean()`. We compute directly from logits, without evaluating $\log(\operatorname{sigmoid}(z))$.


In [2]:
def binary_ce(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Elementwise binary CE from logits; targets may be soft."""
    return F.softplus(logits) - targets * logits


def binary_robust_ce(
    logits: torch.Tensor,
    targets: torch.Tensor,
    delta: float,
) -> torch.Tensor:
    """Elementwise RobustCE for targets in {0, 1} and scalar delta >= 0."""
    if delta < 0:
        raise ValueError("delta must be nonnegative.")
    delta_hat = min(1.0, delta)
    regularizer = delta_hat * F.relu((2 * targets - 1) * logits)
    return binary_ce(logits, targets) + regularizer

### Independent numerical maximization

Let $q$ be the probability of class 1 in the target distribution $(1-q,q)$. For an observed label $y$, the feasible interval is

$$
q\in[q_{\min},q_{\max}],\qquad
q_{\min}=\max(0,y-\delta),\quad q_{\max}=\min(1,y+\delta).
$$

For fixed $z$ and $q$:

$$
\operatorname{CE}(z,q)
=-(1-q)\log(1-\sigma(z))-q\log\sigma(z)
=\operatorname{softplus}(z)-qz.
$$

We construct **10,001 equally spaced points**, including both interval endpoints, evaluate CE at every $q$, and select the maximum. For an independent implementation, we use the built-in `binary_cross_entropy_with_logits`: the numerical computation does not call our analytic loss function or use an analytic rule for selecting the worst-case target.

If $\delta=0$, the interval reduces to a single point. For $\delta\ge1$, it covers all of $[0,1]$.


In [3]:
N_GRID = 10_001
ATOL = 1e-10


def ce_on_feasible_grid(
    logit: float,
    target: int,
    delta: float,
    n_grid: int = N_GRID,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return feasible q values and their CE losses without an analytic maximizer."""
    if target not in (0, 1):
        raise ValueError("The observed label must be 0 or 1.")
    if delta < 0:
        raise ValueError("delta must be nonnegative.")
    if n_grid < 2:
        raise ValueError("The grid must contain at least two points.")

    q_min = max(0.0, target - delta)
    q_max = min(1.0, target + delta)
    q_grid = torch.linspace(q_min, q_max, n_grid, device=DEVICE, dtype=DTYPE)
    z = torch.full_like(q_grid, float(logit))
    ce_values = F.binary_cross_entropy_with_logits(z, q_grid, reduction="none")
    return q_grid, ce_values

### Selected inputs

We check every combination of:

- $y\in\{0,1\}$;
- $z\in\{-8,-2,0,2,8\}$;
- $\delta\in\{0,0.05,0.2,0.5,1,1.5\}$.

These **60 cases** cover both classes, confident correct and incorrect predictions, a zero logit, a zero radius, and interval saturation. We include $\delta=1.5$ to check radius clipping.

At $z=0$, every feasible $q$ gives the same loss. `argmax` returns the first grid point; this is one maximizer among many.


In [4]:
LABELS = (0, 1)
LOGITS = (-8.0, -2.0, 0.0, 2.0, 8.0)
DELTAS = (0.0, 0.05, 0.2, 0.5, 1.0, 1.5)

results = []
with torch.no_grad():
    for y, z, delta in product(LABELS, LOGITS, DELTAS):
        logits = torch.tensor(z, device=DEVICE, dtype=DTYPE)
        targets = torch.tensor(float(y), device=DEVICE, dtype=DTYPE)
        analytic = binary_robust_ce(logits, targets, delta).item()

        q_grid, ce_values = ce_on_feasible_grid(z, y, delta)
        max_index = int(torch.argmax(ce_values).item())
        numeric = ce_values[max_index].item()

        results.append(
            {
                "y": y,
                "z": z,
                "delta": delta,
                "q_min": q_grid[0].item(),
                "q_max": q_grid[-1].item(),
                "q_numeric": q_grid[max_index].item(),
                "L_analytic": analytic,
                "L_numeric": numeric,
                "abs_error": abs(analytic - numeric),
            }
        )

analytic_values = torch.tensor([r["L_analytic"] for r in results], dtype=DTYPE)
numeric_values = torch.tensor([r["L_numeric"] for r in results], dtype=DTYPE)

# Compare maximum values: the maximizer is not unique at z=0.
torch.testing.assert_close(analytic_values, numeric_values, rtol=0, atol=ATOL)

max_abs_error = max(r["abs_error"] for r in results)
display(
    Markdown(
        f"**All {len(results)} cases passed.** "
        f"Maximum absolute error: **{max_abs_error:.3e}**, "
        f"tolerance: **{ATOL:.0e}** (relative tolerance is zero)."
    )
)

**All 60 cases passed.** Maximum absolute error: **6.661e-16**, tolerance: **1e-10** (relative tolerance is zero).

### Visualizing the maximization

For $\delta=0.2$, we plot CE as a function of $q$ at $z=-2,0,2$ for both observed labels. The gray line shows CE over all of $[0,1]$; the blue segment highlights the feasible interval. The dashed line shows the analytic RobustCE value, and the orange point marks a numerically selected maximum. At $z=0$, every point in the feasible interval is a maximizer.

The Plotly figure is interactive: hover for exact values, zoom into a panel, and click legend entries to toggle a trace group.


In [5]:
plot_delta = 0.2
plot_logits = (-2.0, 0.0, 2.0)
fig = make_subplots(
    rows=2,
    cols=3,
    shared_xaxes=True,
    shared_yaxes=True,
    subplot_titles=[f"y = {y}, z = {z:g}" for y in LABELS for z in plot_logits],
    vertical_spacing=0.18,
    horizontal_spacing=0.07,
)

with torch.no_grad():
    for row, y in enumerate(LABELS, start=1):
        for col, z in enumerate(plot_logits, start=1):
            q_all = torch.linspace(0, 1, 101, device=DEVICE, dtype=DTYPE)
            ce_all = F.binary_cross_entropy_with_logits(
                torch.full_like(q_all, z), q_all, reduction="none"
            )
            q_grid, ce_grid = ce_on_feasible_grid(z, y, plot_delta)
            max_index = int(torch.argmax(ce_grid).item())
            analytic = binary_robust_ce(
                torch.tensor(z, device=DEVICE, dtype=DTYPE),
                torch.tensor(float(y), device=DEVICE, dtype=DTYPE),
                plot_delta,
            ).item()

            # Search uses all 10,001 points; plotting a linear function needs fewer.
            plot_indices = torch.linspace(0, len(q_grid) - 1, 101).long()
            traces = [
                go.Scatter(
                    x=q_all.tolist(), y=ce_all.tolist(),
                    mode="lines", name="CE on [0, 1]",
                    legendgroup="full", line=dict(color="#b6bec8", width=2),
                ),
                go.Scatter(
                    x=q_grid[plot_indices].tolist(), y=ce_grid[plot_indices].tolist(),
                    mode="lines", name="Feasible q",
                    legendgroup="feasible", line=dict(color="#2563eb", width=4),
                ),
                go.Scatter(
                    x=[0, 1], y=[analytic, analytic],
                    mode="lines", name="Analytic RobustCE",
                    legendgroup="analytic",
                    line=dict(color="#16a34a", width=2, dash="dash"),
                ),
                go.Scatter(
                    x=[q_grid[max_index].item()], y=[ce_grid[max_index].item()],
                    mode="markers", name="Numerical maximum",
                    legendgroup="numeric",
                    marker=dict(color="#f97316", size=10),
                ),
            ]
            for trace in traces:
                trace.showlegend = row == 1 and col == 1
                trace.hovertemplate = "q=%{x:.6f}<br>CE=%{y:.8f}<extra>%{fullData.name}</extra>"
                fig.add_trace(trace, row=row, col=col)

            fig.add_vrect(
                x0=q_grid[0].item(), x1=q_grid[-1].item(),
                fillcolor="#2563eb", opacity=0.08, line_width=0,
                layer="below", row=row, col=col,
            )
            if z == 0:
                fig.add_annotation(
                    x=0.5, y=1.65,
                    text="Every feasible q is a maximizer",
                    showarrow=False, font=dict(size=11), row=row, col=col,
                )

fig.update_xaxes(range=[-0.04, 1.04])
fig.update_yaxes(range=[0, 2.3])
fig.update_xaxes(title_text="q: target probability of class 1", row=2)
fig.update_yaxes(title_text="CE(z, q)", col=1)
fig.update_layout(
    title=f"Maximizing CE over the feasible target interval: delta = {plot_delta}",
    height=700,
    margin=dict(l=65, r=25, t=100, b=120),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.20),
    hovermode="closest",
)
fig.show(config={"displaylogo": False, "responsive": True})


### Interpreting the check

For a fixed logit, CE is linear in $q$, so its maximum over a closed interval is attained at an endpoint. Because the numerical grid includes both endpoints, we expect agreement up to floating-point roundoff despite using a finite grid. At $z=0$, the function is constant and the maximizer is not unique.

When a prediction is confident and correct, shifting the target within the feasible interval increases CE, as captured by the additional term $R$. When the logit's sign contradicts the observed label, the observed label itself remains a worst-case feasible target, and the additional term is zero.

This check confirms that the two ways of computing the loss agree **at the selected inputs**. The execution result and maximum absolute error are reported above.
